In [ ]:
# Bug #5130 Reproduction: Zonemap Index Scanning Too Many Fragments
# 
# Run this notebook with: LANCE_LOG=debug jupyter notebook
# Or set environment variable before starting Jupyter

import lance
import pyarrow as pa

print("=" * 80)
print("BUG #5130: Zonemap Index Scanning Too Many Fragments")
print("=" * 80)
print("\nNote: Run with LANCE_LOG=debug to see execution plans")
print("The key metric is 'num_fragments' in the execution plan.\n")

✓ Tracing enabled


In [3]:
# Create dataset with 10 fragments, each fragment has unique test_id
# This simulates the bug scenario where each fragment has only one test_id value

fragments_data = []
num_fragments = 10
rows_per_fragment = 100

for fragment_idx in range(num_fragments):
    # Each fragment gets a unique test_id
    test_id = f"test_id_{fragment_idx}"
    
    # Create rows for this fragment - all rows have the SAME test_id
    fragment_data = pa.table({
        "id": range(fragment_idx * rows_per_fragment, (fragment_idx + 1) * rows_per_fragment),
        "test_id": [test_id] * rows_per_fragment,  # All rows in fragment have same test_id
        "value": [fragment_idx] * rows_per_fragment,
    })
    fragments_data.append(fragment_data)

# Combine all fragments into one table
full_data = pa.concat_tables(fragments_data)

print(f"Creating dataset with {len(full_data)} rows")
print(f"Sample data:")
print(full_data.slice(0, 5).to_pandas())
print("...")
print(full_data.slice(95, 10).to_pandas())  # Show boundary between fragments

# Write dataset with max_rows_per_file to create multiple fragments
# This ensures each fragment has unique test_id values
ds = lance.write_dataset(
    full_data, 
    "./zonemap_bug.lance", 
    max_rows_per_file=rows_per_fragment,  # One fragment per batch
    max_rows_per_group=10,  # Smaller row groups for better zonemap granularity
    mode="overwrite"
)

# Verify fragment count
fragments = ds.get_fragments()
print(f"\n✓ Created {len(fragments)} fragments")

# Show what test_id values are in each fragment
print("\nFragment distribution:")
for i, frag in enumerate(fragments):
    sample = frag.head(1).to_pydict()
    print(f"  Fragment {i}: test_id = {sample['test_id'][0]}")




Creating dataset with 1000 rows
Sample data:
   id    test_id  value
0   0  test_id_0      0
1   1  test_id_0      0
2   2  test_id_0      0
3   3  test_id_0      0
4   4  test_id_0      0
...
    id    test_id  value
0   95  test_id_0      0
1   96  test_id_0      0
2   97  test_id_0      0
3   98  test_id_0      0
4   99  test_id_0      0
5  100  test_id_1      1
6  101  test_id_1      1
7  102  test_id_1      1
8  103  test_id_1      1
9  104  test_id_1      1

✓ Created 10 fragments

[2025-11-06T18:33:07Z WARN  lance::dataset::write::insert] No existing dataset at ./zonemap_bug.lance, it will be created




Fragment distribution:
  Fragment 0: test_id = test_id_0
  Fragment 1: test_id = test_id_1
  Fragment 2: test_id = test_id_2
  Fragment 3: test_id = test_id_3
  Fragment 4: test_id = test_id_4
  Fragment 5: test_id = test_id_5
  Fragment 6: test_id = test_id_6
  Fragment 7: test_id = test_id_7
  Fragment 8: test_id = test_id_8
  Fragment 9: test_id = test_id_9


In [ ]:
# First, query WITHOUT index to establish baseline
print("=" * 80)
print("QUERY WITHOUT INDEX (BASELINE)")
print("=" * 80)

result_no_index = ds.to_table(filter="test_id = 'test_id_5'")
print(f"✓ Query returned {len(result_no_index)} rows (expected: 100)")
print("\n⚠️  Check logs for: num_fragments=10 (full table scan expected)")

# Create zonemap index on test_id column
print("\n" + "=" * 80)
print("CREATING ZONEMAP INDEX")
print("=" * 80)

ds.create_scalar_index("test_id", index_type="ZONEMAP")

indices = ds.list_indices()
print(f"✓ Created index: {[idx['name'] for idx in indices]}")

# Query WITH index - THIS IS WHERE THE BUG MANIFESTS
print("\n" + "=" * 80)
print("QUERY WITH ZONEMAP INDEX (BUG CHECK)")
print("=" * 80)
print("Querying for: test_id = 'test_id_5'")
print("Expected: num_fragments=1 (should scan ONLY fragment 5)")
print("-" * 80)

result_with_index = ds.to_table(filter="test_id = 'test_id_5'")

print(f"\n✓ Query returned {len(result_with_index)} rows")
print("\n🐛 BUG #5130 CHECK:")
print("   Look in logs for the execution plan:")
print("   ")
print("   Executing plan:")
print("     LanceRead: ... num_fragments=??? ...")
print("       ScalarIndexQuery: query=[test_id = test_id_5]@test_id_idx")
print("   ")
print("   ⚠️  If num_fragments=10, the BUG IS PRESENT")
print("   ✓  If num_fragments=1, the bug is FIXED")

QUERY WITHOUT INDEX

✓ Query returned 100 rows (expected: 100)
✓ Captured 0 trace events

CREATING ZONEMAP INDEX
✓ Created index: [{'name': 'test_id_idx', 'type': 'ZoneMap', 'uuid': 'ca50f934-ff5d-462f-bf27-672ac63c2791', 'fields': ['test_id'], 'version': 1, 'fragment_ids': {0, 1, 2, 3, 4, 5, 6, 7, 8, 9}, 'base_id': None}]

QUERY WITH ZONEMAP INDEX
Querying for: test_id = 'test_id_5'
Expected: Should scan ONLY fragment 5 (1 fragment)
--------------------------------------------------------------------------------

✓ Query returned 100 rows
✓ Captured 0 trace events

TRACE EVENT ANALYSIS

Captured 0 trace events

All event targets:

LOOKING FOR num_fragments in events...


In [ ]:
# Summary and verification
print("=" * 80)
print("BUG #5130 REPRODUCTION SUMMARY")
print("=" * 80)

print("\n📋 Dataset Configuration:")
print(f"  - Total fragments: {len(ds.get_fragments())}")
print(f"  - Rows per fragment: {rows_per_fragment}")
print(f"  - Each fragment has UNIQUE test_id values")

print("\n🔍 Query: test_id = 'test_id_5'")
print(f"  - Expected: Scan ONLY 1 fragment (fragment 5)")
print(f"  - Actual result: {len(result_with_index)} rows ✓")

# Verify the query correctness
print("\n✅ Query Correctness:")
sample_rows = result_with_index.to_pydict()
unique_test_ids = set(sample_rows['test_id'])
print(f"  - Unique test_ids in result: {unique_test_ids}")
print(f"  - All rows have correct test_id: {unique_test_ids == {'test_id_5'}}")
print(f"  - Returns correct count: {len(result_with_index) == 100}")

print("\n📝 How to Check for the Bug:")
print("  1. Run terminal: LANCE_LOG=debug python script.py 2>&1 | grep 'Executing plan'")
print("  2. Look for the line with 'ScalarIndexQuery'")
print("  3. Check the num_fragments value:")
print("     - num_fragments=10 → BUG IS PRESENT (scanning all fragments)")
print("     - num_fragments=1  → Bug is fixed (only scanning fragment 5)")


BUG #5130 REPRODUCTION SUMMARY

📋 Dataset Configuration:
  - Total fragments: 10
  - Rows per fragment: 100
  - Each fragment has UNIQUE test_id values

🔍 Query: test_id = 'test_id_5'
  - Expected: Scan ONLY 1 fragment (fragment 5)
  - Actual result: 100 rows ✓

🐛 Bug Analysis:
Looking for 'num_fragments' in trace events...
  (num_fragments not found in trace events)
  Note: Trace events might not capture execution plan details
  The query still returns correct results though!

✅ Query Correctness:
  - Unique test_ids in result: {'test_id_5'}
  - All rows have correct test_id: True


In [ ]:
# Additional test: Query multiple test_ids to see the pattern
print("=" * 80)
print("TESTING MULTIPLE QUERIES")
print("=" * 80)

test_queries = ['test_id_0', 'test_id_3', 'test_id_7', 'test_id_9']

print("Running queries for different test_ids...")
for test_id_query in test_queries:
    result = ds.to_table(filter=f"test_id = '{test_id_query}'")
    print(f"  Query '{test_id_query}': returned {len(result)} rows")

print("\n💡 Expected behavior with zonemap index:")
print("   Each query should scan exactly 1 fragment")
print("   (since each fragment contains only one unique test_id value)")
print("\n⚠️  With bug #5130:")
print("   Each query scans ALL 10 fragments (check logs)")
print("\n📊 Production Impact:")
print("   - 30K fragments → 3K+ scanned instead of 1")
print("   - 3000x performance overhead!")


TESTING MULTIPLE QUERIES
? Query 'test_id_0': returned 100 rows (num_fragments not captured)
? Query 'test_id_3': returned 100 rows (num_fragments not captured)
? Query 'test_id_7': returned 100 rows (num_fragments not captured)
? Query 'test_id_9': returned 100 rows (num_fragments not captured)

💡 Expected: Each query should scan exactly 1 fragment
   (since each fragment contains only one unique test_id value)
